In [1]:
import os
import fitz
import numpy as np
import pandas as pd
import requests
import getpass
from datetime import date as dt

In [2]:
username=str(input('Type Your Username:\n'))
print()
password=str(getpass.getpass(prompt='Password'))
crdtstr=str(input('Enter created date start in yyyy-mm-dd:\n'))

Type Your Username:
 mintomathew.jose+ps+leap@arcadia.com


Password ········
Enter created date start in yyyy-mm-dd:
 2025-06-30


In [3]:
url = "https://api.urjanet.com/auth/login"
payloadl = {
    "username": username,
    "password": password
    }
h = {
        "accept": "application/json",
        "content-type": "application/json"
    }
def Api_tkn_request(payloadl):
    
    url = "https://api.urjanet.com/auth/login"
    resd = requests.post(url, json=payloadl, headers=h)
    tkn=resd.json()['token']
    #print(tkn)
    headers = {
      'Accept': 'application/json',
      'Authorization': 'Bearer '+tkn}
    return headers

In [4]:
headers=Api_tkn_request(payloadl)

In [5]:
new_acc=requests.get('https://api.urjanet.com/utility/accounts?search=providerAlias%3D%3DComEd&createdDate=ge='+crdtstr+'&size=100', params={}, headers = headers).json()
pg=new_acc['page']['totalPages']
#new_acc

In [6]:
accdict=list()
nullist=list()
choiceid=np.nan
for j in range(0,pg):
    accs=requests.get('https://api.urjanet.com/utility/accounts?search=providerAlias%3D%3DComEd&createdDate=ge='+crdtstr+'&page='+str(j)+'&size=100', params={}, headers = headers).json()
    for i in accs['_embedded']['accounts']:
        if not i['customData']['Electric_Choice_Id']:
        #if i['customData']['Electric_Choice_Id']:
            #print(i['customData']['Electric_Choice_Id'])
            choiceid=np.nan
            accountnumber=i['accountNumber']
            createddate=i['createdDate']
            statementlink=i['_links']['statements']['href']
            accid=i['entityId']
            statements=requests.get(statementlink, params={}, headers = headers).json()
            #print(statements)
            print("Account Number: "+accountnumber)
            lateststatementid=statements['_embedded']['statements'][0]['entityId']
            url = "https://api.urjanet.com/utility/statements/"+lateststatementid
            response = requests.get(url, headers=headers)
            if response.json()["accountData"][0]['meterData']:
                if response.json()["accountData"][0]['meterData'][0]['podNumber']:
                    choiceid= str(response.json()["accountData"][0]['meterData'][0]['podNumber'])
                    print("Choice ID: "+choiceid)
                    print("\n")
                    newac={'accountnumber':accountnumber,'createddate':createddate,'choiceid':choiceid,'accountid':accid}
                    accdict.append(newac)
                else:
                    newac={'accountnumber':accountnumber,'createddate':createddate,'choiceid':"Choice ID not available",'accountid':accid}
                    nullist.append(newac)
                    print("Electric Choice ID Not Found\n")
            else:
                newac={'accountnumber':accountnumber,'createddate':createddate,'choiceid':"No Meter data available",'accountid':"No Meter data available"}
                print("No Meter data available\n")
            
            #p=requests.get(lateststatementsource, params={}, headers = headers)
            #with open(str(accountnumber)+".pdf", 'wb') as f:
            #    f.write(p.content)
            #doc=fitz.open(str(accountnumber)+".pdf")
            #text=""
            #for page in doc:
            #    text=text+page.get_text()
            #    
            #if text.find("Electric Choice ID:") !=-1:
            #    ind=text.find("Electric Choice ID:")
            #    choiceid=text[ind+len("Electric Choice ID:"):text.find("\n",ind)]
            #    choiceid.removeprefix(" ")
            #    print("Choice ID: "+choiceid)
            #else:
            #    print("Electric Choice ID Not Found")
            #accid=i['entityId']    
            
            #print(newac)
            
ac=pd.DataFrame(accdict)  
ns=pd.DataFrame(nullist)

Account Number: 7657113000
Choice ID: 7651880787


Account Number: 8568692846
Choice ID: 8560688836


Account Number: 0765500100
Choice ID: 0760946670


Account Number: 5135910100
Choice ID: 5130781234


Account Number: 8276591222
Choice ID: 8270691393


Account Number: 2359333000
Choice ID: 2352696291


Account Number: 7758002222
Choice ID: 7753537605


Account Number: 3278988000
Choice ID: 3272816900


Account Number: 8188204000
Choice ID: 8181661787


Account Number: 5378558000
Choice ID: 5370122977


Account Number: 5994885415
Choice ID: 5993098962


Account Number: 6379221222
Choice ID: 6372615451


Account Number: 1754889472
Choice ID: 1754922009


Account Number: 1560917107
Choice ID: 1563836790


Account Number: 4392177000
Choice ID: 4390289887


Account Number: 8730453000
Choice ID: 8730813421


Account Number: 6315153000
Choice ID: 6311483559


Account Number: 4022055000
Choice ID: 4020289225


Account Number: 8472159822
Choice ID: 8470988025


Account Number: 2257780869
Elec

In [7]:
ac.to_excel("Accounts.xlsx")
ns.reset_index(drop=True,inplace=True)
ns.to_excel("Null_report.xlsx")

plc=pd.read_excel("Accounts_PLC.xlsx")

plc.fillna("Null",inplace=True)

plc.reset_index(drop=True,inplace=True)

for i in range(len(plc)):
    if plc['choiceid'][i]!="Null":
        url = "https://api.urjanet.com/utility/accounts/"+plc["accountid"][i]
    
        payload = { "customData": {
                "Electric_Choice_Id": int(plc['choiceid'][i]),
                "Capacity_PLC": plc['Capacity PLC'][i],
                "Capacity_PLC_Start_Date": str(plc['Start date'][i])
            } }
        
        headers=Api_tkn_request(payloadl)
        response = requests.patch(url, json=payload, headers=headers)
        
        print(response.text)
            



In [8]:
dd=input("Press Enter to continue")

Press Enter to continue 


In [9]:
headers=Api_tkn_request(payloadl)

In [10]:
df=ac

In [11]:
l1=[]
for i in range(len(df)):
    try:
        df1=pd.read_table(str(df["choiceid"][i])+".csv")
    except:
        print("No File found for account #"+str(df["accountnumber"][i]))
        print("Updating the values as NULL for account #"+str(df["accountnumber"][i]))
        l1.append({"accountnumber":df["accountnumber"][i],"accountid":df["accountid"][i],"choiceid":df["choiceid"][i],"plc1":"PLC Not Found","strt1":"PLC Start Date Not Found","end1":"PLC End Date Not Found","plc2":"PLC Not Found","strt2":"PLC Start Date Not Found","end2":"PLC End Date Not Found"})
        continue
    #plc1,strt1,end1=str(df1['Summary Data for Electric Choice ID,'+str(int(df["choiceid"][i]))][2]).split()
    #plc2,strt2,end2=str(df1['Summary Data for Electric Choice ID,'+str(int(df["choiceid"][i]))][3]).split()
    
    if df1['Summary Data for Electric Choice ID,'+str((df["choiceid"][i]))][1]=='Capacity PLC,Start Date,End Date':
        plc1,strt1,end1=str(df1['Summary Data for Electric Choice ID,'+str((df["choiceid"][i]))][2]).split(",")
        plc1=float(plc1.removeprefix("'"))
        #strt1=p1[1]
        end1=end1.removesuffix("'")
        
        plc2,strt2,end2=str(df1['Summary Data for Electric Choice ID,'+str((df["choiceid"][i]))][4]).split(",")
        plc2=float(plc2.removeprefix("'"))
        #strt2=p2[1]
        end2=end2.removesuffix("'")
        
        l1.append({"accountnumber":df["accountnumber"][i],"accountid":df["accountid"][i],"choiceid":df["choiceid"][i],"plc1":plc1,"strt1":strt1,"end1":end1,"plc2":plc2,"strt2":strt2,"end2":end2})
        
    else:
        print("PLC value not found in the file "+str(int(df["choiceid"][i]))+".csv")
        l1.append({"accountnumber":df["accountnumber"][i],"accountid":df["accountid"][i],"choiceid":df["choiceid"][i],"plc1":"PLC Data not available","strt1":"","end1":"","plc2":"PLC Data not available","strt2":"","end2":""})

No File found for account #8568692846
Updating the values as NULL for account #8568692846
No File found for account #7758002222
Updating the values as NULL for account #7758002222
No File found for account #8472159822
Updating the values as NULL for account #8472159822


In [12]:
df2=pd.DataFrame(l1)
df2.to_excel("Accounts_PLC.xlsx")
df2.fillna("Null",inplace=True)
df2.reset_index(drop=True,inplace=True)
df2

,accountnumber,accountid,choiceid,plc1,strt1,end1,plc2,strt2,end2
0,7657113000,1f05c8a2-722f-d542-b75f-e6ab1190ca36,7651880787,4.0361,06/01/2025,05/31/2026,4.9027,01/01/2025,12/31/2025
1,8568692846,1f05b9f4-edf2-d594-9af6-d64a5a52c996,8560688836,PLC Not Found,PLC Start Date Not Found,PLC End Date Not Found,PLC Not Found,PLC Start Date Not Found,PLC End Date Not Found
2,0765500100,1f05b1a1-df04-dc6c-b028-4eab08bc5398,0760946670,3.9228,06/01/2025,05/31/2026,4.2619,01/01/2025,12/31/2025
3,5135910100,1f05ace3-ec22-d996-a281-ce421fe8687d,5130781234,4.7213,06/01/2025,05/31/2026,5.0979,01/01/2025,12/31/2025
4,8276591222,1f05acc4-8397-daf8-a281-ce421fe8687d,8270691393,1.3678,06/01/2025,05/31/2026,1.0816,01/01/2025,12/31/2025
5,2359333000,1f05ab40-d534-d7d4-a281-ce421fe8687d,2352696291,2.7625,06/01/2025,05/31/2026,3.1781,01/01/2025,12/31/2025
6,7758002222,1f05a96a-e894-d0d3-9a62-1a1c384d7127,7753537605,PLC Not Found,PLC Start Date Not Found,PLC End Date Not Found,PLC Not Found,PLC Start Date Not Found,PLC End Date Not Found
7,3278988000,1f05a090-85a3-d438-a777-8a5cbe25d511,3272816900,3.1376,06/01/2025,05/31/2026,3.0784,01/01/2025,12/31/2025
8,8188204000,1f059da4-46aa-decd-a777-8a5cbe25d511,8181661787,3.3026,06/01/2025,05/31/2026,4.4554,01/01/2025,12/31/2025
9,5378558000,1f058720-f947-d9e0-b76b-b2fc9f08a2a8,5370122977,1.8019,06/01/2025,05/31/2026,2.3473,01/01/2025,12/31/2025


In [13]:
for i in range(len(df2)):
    if df2.choiceid[i] !="Choice Id not available":
        url = "https://api.urjanet.com/utility/accounts/"+df2["accountid"][i]
        
        payload = { "customData": {
                "Electric_Choice_Id": str(df2['choiceid'][i]),
                "Capacity_PLC_1": df2['plc1'][i],
                "Capacity_PLC_Start_Date_1": str(df2['strt1'][i]),
                "Capacity_PLC_2": df2['plc2'][i],
                "Capacity_PLC_Start_Date_2": str(df2['strt2'][i])
            } }
        
        headers=Api_tkn_request(payloadl)
        response = requests.patch(url, json=payload, headers=headers)
        
        print(response.text)
        
        

{
  "accountNumber" : "7657113000",
  "createdBy" : "system",
  "createdDate" : "2025-07-09T06:01:28.000+00:00",
  "customData" : {
    "Capacity_PLC_Start_Date_2" : "01/01/2025",
    "Capacity_PLC_Start_Date_1" : "06/01/2025",
    "Electric_Choice_Id" : "7651880787",
    "Capacity_PLC_2" : "4.9027",
    "Capacity_PLC_1" : "4.0361"
  },
  "enabled" : true,
  "entityId" : "1f05c8a2-722f-d542-b75f-e6ab1190ca36",
  "firstExtracted" : "2025-07-09T06:01:28.000+00:00",
  "lastModifiedBy" : "mintomathew.jose+ps+leap@arcadia.com",
  "lastModifiedDate" : "2025-07-09T08:10:51.825+00:00",
  "lastSuccessfulExtraction" : null,
  "latestNewStatement" : "2025-07-09T06:01:29.000+00:00",
  "latestStatementDate" : "2025-06-13",
  "meterCount" : 1,
  "nextExpectedPostDate" : "2025-07-13",
  "normalizedAccountNumber" : "7657113000",
  "origin" : null,
  "prepaid" : false,
  "provider" : {
    "country" : "USA",
    "entityId" : "36dc3aa3-ea0a-11e0-b6bb-12313d2b6294",
    "providerName" : "Commonwealth Edi